# Zoo Animal Type Classification

101 animals, each described by 16 yes/no traits — hair, feathers, eggs, milk, airborne, aquatic, predator, toothed, backbone, breathes, venomous, fins, legs, tail, domestic, catsize — plus a name. Predict the animal's type from its traits alone.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Zoo (id 111)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/111/data.csv')))
print(df.shape)
print('Animals per type code:')
print(df['type'].value_counts().sort_index())
df.head(3)


(101, 18)
Animals per type code:
type
1    41
2    20
3     5
4    13
5     4
6     8
7    10
Name: count, dtype: int64


,animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,type
0,aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
1,antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
2,bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4


In [2]:

feats = [c for c in df.columns if c not in ('animal_name', 'type')]
names = {1: 'mammal', 2: 'bird', 3: 'reptile', 4: 'fish', 5: 'amphibian', 6: 'insect', 7: 'invertebrate'}
X = df[feats]; y = df['type'].map(names)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9231
Logistic Regression accuracy: 0.9231
              precision    recall  f1-score   support

   amphibian       0.00      0.00      0.00         1
        bird       0.83      1.00      0.91         5
        fish       1.00      1.00      1.00         3
      insect       1.00      1.00      1.00         2
invertebrate       1.00      1.00      1.00         3
      mammal       1.00      1.00      1.00        11
     reptile       0.00      0.00      0.00         1

    accuracy                           0.92        26
   macro avg       0.69      0.71      0.70        26
weighted avg       0.89      0.92      0.91        26



/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which traits separate the animal types?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


milk        0.138924
eggs        0.108566
feathers    0.104122
toothed     0.098344
legs        0.086142
hair        0.082764
breathes    0.070205
backbone    0.064753
fins        0.063183
tail        0.056143
airborne    0.051165
aquatic     0.029744
venomous    0.016620
catsize     0.016074
predator    0.012341
domestic    0.000911
dtype: float64


Winner: Random Forest
